In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
ARTIFACTS_DIR = Path.cwd().parent / "artifacts"
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
TOP_K = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
embeddings = np.load(ARTIFACTS_DIR / "embeddings.npy")

metadata = pd.read_parquet(ARTIFACTS_DIR / "metadata.parquet")

config = json.load(open(ARTIFACTS_DIR / "embedding_config.json", "r"))

print("Embeddings:", embeddings.shape)
print("Metadata:", metadata.shape)
print("Embedding model:", config["embedding_model"])

In [ ]:
embedding_model = SentenceTransformer(config["embedding_model"], device = DEVICE)
print("Embedding model loaded:", embedding_model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype = "auto", device_map = "auto")
llm_model.eval()

In [ ]:
def retrieve(query, top_k = 5):
    query_embedding = embedding_model.encode([query], convert_to_numpy = True, normalize_embeddings = True)
    scores = cosine_similarity(query_embedding, embeddings)[0]

    top_k = np.argsort(scores)[::-1][:top_k]
    results = metadata.iloc[top_k].copy()

    results["similarity"] = scores[top_k]

    return results.reset_index(drop = True)

In [ ]:
TEXT_COLUMN = "document"

if TEXT_COLUMN not in metadata.columns:
    raise ValueError(
        f"Expected '{TEXT_COLUMN}' column, "
        f"but found: {metadata.columns.tolist()}"
    )

In [ ]:
def build_context(results):

    context_parts = []

    for i, row in results.iterrows():

        context_parts.append(
            f"""
SOURCE {i + 1}
Similarity: {row["similarity"]:.4f}

{row[TEXT_COLUMN]}
"""
        )

    return "\n".join(context_parts)

In [ ]:
SYSTEM_PROMPT = """
You are MIRx, a medical information retrieval assistant.

Your task is to answer questions using ONLY the information
provided in the retrieved context.

Rules:

1. Do not invent information.
2. Do not introduce medical facts that are absent from the context.
3. If the retrieved context is insufficient, say so clearly.
4. Keep the answer concise and well structured.
5. When possible, mention the relevant drug or medical entity.
6. Do not make diagnoses or provide personalized medical treatment.
7. Clearly distinguish information found in the retrieved context
   from general uncertainty.
"""

In [ ]:
def generate_with_qwen(query, context):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Retrieved medical information:

{context}

User question:

{query}

Using only the retrieved medical information above,
answer the user's question.
"""
        }
    ]

    # Create model inputs
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Move inputs to the model device
    inputs = inputs.to(
        next(llm_model.parameters()).device
    )

    # Number of input tokens
    input_length = inputs["input_ids"].shape[-1]

    # Generate
    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    # Extract only newly generated tokens
    generated_tokens = outputs[
        0,
        input_length:
    ]

    # Decode
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
def generate_response(query, top_k = 5):

    # -------------------------
    # 1. Retrieve
    # -------------------------

    results = retrieve(query, top_k = top_k)

    # -------------------------
    # 2. Build context
    # -------------------------

    context = build_context(results)

    # -------------------------
    # 3. Generate answer
    # -------------------------

    answer = generate_with_qwen(query, context)

    return answer, results

In [ ]:
query = "What are the alternatives to paracetamol?"

answer, sources = generate_response(query, top_k = 5)

print("=" * 80)
print("MIRx RESPONSE")
print("=" * 80)

print(answer)

print("\n")
print("=" * 80)
print("RETRIEVED SOURCES")
print("=" * 80)

for i, row in sources.iterrows():

    print(
        f"\n[{i + 1}] "
        f"Similarity: {row['similarity']:.4f}"
    )

    print(row["document"])

In [ ]:
print(metadata.columns.tolist())

In [ ]:
display(metadata.head())

In [ ]:
test_queries = [
    "What are the alternatives to paracetamol?",
    "What are the side effects of ibuprofen?",
    "Which drugs are used for pain relief?",
    "What are the alternatives to aspirin?",
    "Which medicines are associated with nausea?"
]

for query in test_queries:

    print("\n")
    print("#" * 100)
    print("QUERY:", query)

    answer, sources = generate_response(
        query,
        top_k=5
    )

    print("\nANSWER:")
    print(answer)